<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/langchain_business_usecase_Ollama_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Support Assistant with LangChain

We build a small support chatbot for an online shop called **Northwind Gadgets**.
Customers ask two kinds of questions:

1. **Policy questions** ("What is your return window?") -> the assistant *searches the store's policy documents*.
2. **Order questions** ("Where is order A101?") -> the assistant *looks up the order* in a small order database.

The assistant also **remembers the conversation**, so a customer can ask "can I return *it*?" without repeating the order number.

| Concept | What it means in plain words | Where we use it |
|---|---|---|
| Chunking | Cut long text into small pieces | Step 1 |
| Embeddings | Turn text into numbers that capture its meaning | Step 2 |
| Tools | Python functions the AI is allowed to call | Step 3 |
| Agent | The AI that decides which tool to use | Step 4 |
| Memory | The AI remembers earlier messages | Step 4 and 5 |

**Cost:** everything is free and runs **on your own computer** using [Ollama](https://ollama.com) - no account, no API key, no internet needed after the models are downloaded.

**Works in Google Colab and on your own computer.** Step 0 installs and starts Ollama for you if it is not already running (in Colab, Ollama runs *inside* the Colab machine, not on your laptop).

- **Colab tip:** choose **Runtime -> Change runtime type -> T4 GPU** first. Answers are much faster than on CPU.
- `llama3.1:8b` is the chat model (about 5 GB download). `nomic-embed-text` is the small model that turns text into numbers (about 270 MB).

> **Not a coder?** Read the grey comments (lines starting with `#`) and look at the printed output under each cell.

## Step 0 - Setup

### Cell: install libraries
- **What it does:** downloads the Python libraries this notebook needs (takes 1-2 minutes the first time).
- **Input:** nothing.
- **Output:** a few install messages (or nothing if already installed).

In [ ]:
# "%pip install" downloads libraries from the internet. "-q" keeps the output quiet.
# langchain                -> the main framework (agents, tools)
# langchain-ollama         -> connects LangChain to the models running in Ollama on your computer
# langchain-text-splitters -> cuts long text into chunks
# langgraph                -> gives the agent its memory
# numpy                    -> maths library the search step uses to compare meanings
%pip install -q langchain langchain-ollama langchain-text-splitters langgraph numpy

### Cell: start Ollama and download the two models
- **What it does:** checks whether Ollama is running. If not (for example in a fresh Colab), it installs and starts it. Then it downloads the two models (instant if you already have them).
- **Input:** nothing.
- **Output:** "Ollama is running" and "Models ready". The first run in Colab takes about 3-6 minutes.

In [ ]:
import shutil        # checks whether a program (ollama) is installed
import subprocess    # runs command-line programs from Python
import time          # lets us wait a few seconds
import urllib.request  # used to check whether Ollama answers

def ollama_is_running():
    # Ollama listens on this local address; if it replies, it is running.
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        return True
    except Exception:
        return False

if not ollama_is_running():
    if shutil.which("ollama") is None:
        # Not installed (typical in Colab): install it. zstd is needed by the installer.
        subprocess.run("apt-get install -y -q zstd > /dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True)
    # Start the Ollama server in the background, then wait up to 30 seconds for it.
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_is_running():
            break
        time.sleep(1)

print("Ollama is running:", ollama_is_running())

# Download the chat model and the embedding model (skipped quickly if already present).
for model_name in ["llama3.1:8b", "nomic-embed-text"]:
    subprocess.run(["ollama", "pull", model_name], check=True)
print("Models ready.")

Ollama is running: True
Models ready.


## Step 1 - Prepare the store documents

### Cell: write the documents and cut them into chunks
- **What it does:** defines four short policy documents and cuts each into small pieces ("chunks"). Small chunks make searching more accurate.
- **Input:** `store_documents` (a list of 4 policy texts).
- **Output:** how many chunks were created, and the first chunk as an example.

In [ ]:
# Splits text into pieces without cutting in the middle of sentences where possible.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# In a real project these would come from PDFs or a help-centre. Here they are plain text.
store_documents = [
    "Return Policy: Items can be returned within 30 days of delivery for a full refund, "
    "if unused and in original packaging. Sale items are final and cannot be returned. "
    "Refunds go to the original payment method within 5-7 business days.",

    "Shipping Policy: Standard shipping takes 3-5 business days within the country. "
    "Express shipping (2 business days) is available at checkout for an extra fee. "
    "We do not currently ship internationally.",

    "Warranty Policy: All electronics carry a 1-year manufacturer warranty covering defects, "
    "not accidental damage. Contact support with your order ID to start a warranty claim.",

    "Cancellation Policy: Orders can be cancelled for free within 1 hour of placing them, "
    "as long as they have not shipped. After that, treat it as a return once received.",
]

# chunk_size = maximum characters per chunk, chunk_overlap = characters repeated between neighbours.
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.create_documents(store_documents)   # list of small text pieces

print(f"{len(store_documents)} documents were cut into {len(chunks)} chunks")
print("First chunk:", chunks[0].page_content)

4 documents were cut into 4 chunks
First chunk: Return Policy: Items can be returned within 30 days of delivery for a full refund, if unused and in original packaging. Sale items are final and cannot be returned. Refunds go to the original payment method within 5-7 business days.


## Step 2 - Make the chunks searchable (embeddings)

An **embedding** turns a piece of text into a list of numbers that represents its *meaning*.
Texts with similar meaning get similar numbers, so the computer can find "the chunk closest in meaning to the question" even when the words differ.

### Cell: build the search index
- **What it does:** converts every chunk into numbers and stores them in a small in-memory "vector store". Then it tests the search with one question.
- **Input:** `chunks` from Step 1, and a test question.
- **Output:** the policy chunk that best matches the question (here: the Return Policy).
- **Note:** this uses the `nomic-embed-text` model you pulled in Ollama (runs on your computer).

In [ ]:
# The embedding model, running locally inside Ollama.
from langchain_ollama import OllamaEmbeddings
# A simple store that keeps the embedded chunks in memory (disappears when the notebook stops).
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)                       # embed every chunk and store it

# A "retriever" is the search function. k=2 means: return the 2 best-matching chunks.
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Quick test: note the question never uses the word "policy" - it matches by meaning.
test = retriever.invoke("How long do I have to return something?")
print("Best match:", test[0].page_content)

Best match: Return Policy: Items can be returned within 30 days of delivery for a full refund, if unused and in original packaging. Sale items are final and cannot be returned. Refunds go to the original payment method within 5-7 business days.


## Step 3 - Create the tools

A **tool** is an ordinary Python function that the AI is allowed to call.
The AI reads each tool's *description* (the text in triple quotes) to decide when to use it - so write it clearly.

### Cell: define two tools
- **Tool 1 `search_store_policies`:** input = a question (text); output = the 2 most relevant policy chunks.
- **Tool 2 `get_order_status`:** input = an order ID like `"A101"`; output = that order's details (or "not found").
- **Output of this cell:** the names of the two tools.

In [ ]:
# "@tool" turns a normal Python function into a tool the AI can call.
from langchain.tools import tool


@tool
def search_store_policies(question: str) -> str:
    """Search the store's policy documents (returns, shipping, warranty, cancellation) to answer a customer question."""
    results = retriever.invoke(question)                       # search by meaning
    return "\n".join(r.page_content for r in results)          # join the matching chunks into one text


# A tiny fake order "database" for this demo. In real life this would be a call to your order system.
ORDERS = {
    "A101": {"status": "Shipped",    "delivered_days_ago": 5,    "item": "Wireless Mouse"},
    "A102": {"status": "Processing", "delivered_days_ago": None, "item": "Mechanical Keyboard"},
    "A103": {"status": "Delivered",  "delivered_days_ago": 40,   "item": "USB-C Hub"},
}


@tool
def get_order_status(order_id: str) -> str:
    """Look up an order by its ID (for example 'A101'). Returns status, days since delivery and item name."""
    order = ORDERS.get(order_id.upper())                       # .upper() so "a101" also works
    return str(order) if order else f"No order found with ID {order_id}."


tools = [search_store_policies, get_order_status]
print("Tools ready:", [t.name for t in tools])

Tools ready: ['search_store_policies', 'get_order_status']


## Step 4 - Build the agent (the "brain") with memory

An **agent** is an AI model that can choose which tool to call, call it, read the result, and write the final answer.
**Memory** is added with a *checkpointer*, which saves the conversation under a `thread_id` (think: one chat window per customer).

### Cell: create the agent
- **What it does:** connects to the AI model running on your computer and combines it with the tools and the memory.
- **Input:** the two tools and the instructions (`SYSTEM_PROMPT`).
- **Output:** "Agent ready."
- **If you see a "connection refused" error:** the Ollama app is not running - start it and re-run this cell.
- **Note:** answers may take 10-60 seconds on a laptop without a graphics card. That is normal.

In [ ]:
# ChatOllama is the connector to a chat model running in Ollama on your computer.
from langchain_ollama import ChatOllama
# Builds the agent (model + tools + instructions + memory).
from langchain.agents import create_agent
# Memory that keeps the conversation in RAM (lost when the notebook stops).
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOllama(
    model="llama3.1:8b",    # free open-source model that can use tools (pulled earlier)
    temperature=0,          # 0 = consistent answers, higher = more creative
)

# The instructions that tell the AI who it is and when to use each tool.
SYSTEM_PROMPT = (
    "You are a helpful, concise customer support assistant for Northwind Gadgets. "
    "Use search_store_policies for policy questions and get_order_status for questions "
    "about a specific order. Use both when a question needs both."
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),                  # this line gives the agent memory
)
print("Agent ready.")

Agent ready.


## Step 5 - Chat with the assistant

### Cell: a three-question conversation
- **What it does:** defines a helper `ask(question, thread_id)` that sends one customer message and prints the reply. Then it asks three questions in the *same* chat (`thread_id="customer-1"`).
- **Input:** the customer's question text, and a `thread_id` that identifies the chat.
- **Output:** each question followed by the assistant's answer.
- **What to notice:**
  1. Question 1 needs the **order tool**.
  2. Question 2 says only "it" - the assistant must **remember** order A101, and use **both tools** (the order's delivery date + the return policy).
  3. Question 3 needs only the **policy tool**.

In [ ]:
def ask(question, thread_id):
    # Send the customer's message. Same thread_id = same conversation = shared memory.
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        {"configurable": {"thread_id": thread_id}},
    )
    answer = result["messages"][-1].content        # the last message is the assistant's reply
    print(f"Customer:  {question}\nAssistant: {answer}\n")


ask("Hi, where is my order A101?", "customer-1")
ask("If I don't like it, can I still return it?", "customer-1")   # "it" = order A101 (memory)
ask("Do you ship internationally?", "customer-1")                 # policy question only

Customer:  Hi, where is my order A101?
Assistant: Your order A101 has been shipped and was delivered 5 days ago. It contained a Wireless Mouse.

Customer:  If I don't like it, can I still return it?
Assistant: According to our return policy, you can return the Wireless Mouse within 30 days of delivery, as long as it's unused and in its original packaging. If you'd like to initiate a return, please contact our customer support team for further assistance.

Customer:  Do you ship internationally?
Assistant: Unfortunately, we do not currently ship internationally. If you're interested in purchasing from us, you'll need to have a shipping address within our supported countries.



## Step 6 - Proof that memory matters

### Cell: same follow-up question, but a brand-new chat
- **What it does:** asks the same "can I return it?" question using a **different** `thread_id`, so there is no earlier conversation.
- **Input:** the question and `thread_id="customer-2"`.
- **Output:** the assistant cannot know what "it" is. It gives only a general policy answer and never mentions order A101 or the Wireless Mouse (compare with Step 5, question 2). Exact wording varies from run to run.

In [ ]:
ask("If I don't like it, can I still return it?", "customer-2")   # new chat = no memory of A101

Customer:  If I don't like it, can I still return it?
Assistant: Based on our store policies, if you don't like an item, you can return it within 30 days of delivery as long as it's unused and in its original packaging. Please note that sale items are final and cannot be returned. If you'd like to initiate a return, please contact our customer support team for further assistance.



## Recap

- **Chunking:** `RecursiveCharacterTextSplitter` cut the documents into small pieces.
- **Embeddings + search:** `OllamaEmbeddings` and `InMemoryVectorStore` let us find text by meaning.
- **Tools:** `search_store_policies` and `get_order_status` are normal Python functions with clear descriptions.
- **Agent:** `create_agent` lets the AI choose the right tool for each question.
- **Memory:** the same `thread_id` keeps the conversation; a new `thread_id` starts fresh.

**Try it yourself:** add a new order to `ORDERS`, add a new policy to `store_documents`, or change the `SYSTEM_PROMPT` tone (for example "reply in one sentence") and re-run from that cell.